# Exercise 2.7: Transforming and Merging (Angola IEA and INE trade)

Two sources, two jobs. The survey cleaned in 2.6 becomes analysis ready through
custom functions and `apply`. The INE trade workbooks are then loaded, merged and
stacked.

**PT:** Duas fontes, dois trabalhos. O inquerito limpo em 2.6 torna-se pronto
para analise com funcoes proprias e `apply`. Depois carregamos, juntamos e
empilhamos os ficheiros de comercio do INE.

> **Pipeline:** run 2.6 first. Reads `10_cleaned/` and `0_raw/angola`, writes
> `20_processed/`.

### Path Setup (run first)

**PT:** Configuracao dos caminhos.

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'
DATA_PROC_DIR = '../../data/20_processed'

TRADE_DIR = 'international_trade'

clean_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')
trade_dir = os.path.join(DATA_RAW_DIR, TRADE_DIR)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Trade workbooks available / Ficheiros de comercio disponiveis:')
for name in sorted(os.listdir(trade_dir)):
    print('  ', name)

---

# Part A: the survey

## Task 1: Load the survey and restore its types

CSV forgets dtypes. Notebook 2.6 saved them in the codebook, so read that file
first and use it.

**What to do:** read the codebook, build two dictionaries from it, one mapping
`new_name` to `description` and one mapping `new_name` to `dtype_final`, then
pass the dtypes to `read_csv`. Datetime columns cannot go through `dtype=`, so
they are separated into `parse_dates`.

**PT:** O CSV esquece os tipos. O caderno 2.6 gravou-os no dicionario.

**O que fazer:** leia o dicionario, construa dois dicionarios a partir dele, um
de `new_name` para `description` e outro de `new_name` para `dtype_final`, e
passe os tipos ao `read_csv`. As colunas de data nao podem ir em `dtype=`, por
isso vao em `parse_dates`.

In [ ]:
codebook_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_codebook.csv')
codebook_df = pd.read_csv(codebook_path)

descriptions =   # your code here: new_name -> description
dtypes =   # your code here: new_name -> dtype_final

# dtype= cannot build a datetime, those columns need parse_dates instead
# dtype= nao constroi datas, essas colunas precisam de parse_dates
date_cols = [col for col, kind in dtypes.items() if kind.startswith('datetime')]
read_dtypes = {col: kind for col, kind in dtypes.items()
               if not kind.startswith('datetime')}

df = pd.read_csv(clean_path, dtype=read_dtypes, parse_dates=date_cols)

print('Survey:', df.shape)
print(df[['household_id', 'age', 'job_start_year', 'interview_date']].dtypes)

**Questions:**

- How many rows and columns did you load, and did every dtype come back?
- Which column could not be restored through `dtype=`, and why not?

**PT:** Quantas linhas e colunas carregou, e todos os tipos voltaram? Que coluna
nao pode ser restaurada por `dtype=`, e porque?

---

## Task 2: Write a function and apply it to one column

`apply` on a Series runs your function once per value. Use it when the rule needs
branching that a single expression cannot express clearly.

**What to do:** complete `age_band` so it returns `'Child'` under 15, `'Youth'`
under 25, `'Adult'` under 65 and `'Elderly'` otherwise, then apply it to `age` and
store the result in a new column `age_band`.

**PT:** `apply` numa Serie corre a funcao para cada valor.

**O que fazer:** complete `age_band` para devolver `'Child'` abaixo de 15,
`'Youth'` abaixo de 25, `'Adult'` abaixo de 65 e `'Elderly'` nos restantes casos.
Depois aplique a coluna `age` e guarde em `age_band`.

In [ ]:
def age_band(age):
    """ILO oriented age band for one person / Faixa etaria para uma pessoa."""
    # your code here: band at 15, 25 and 65
    # o seu codigo aqui: faixas em 15, 25 e 65
    return


df['age_band'] = df['age'].apply(age_band)
df['age_band'].value_counts()

**Questions:**

- How many people fall in each band?
- Where do the thresholds 15 and 65 come from?

**PT:** Quantas pessoas em cada faixa? De onde vem os limiares 15 e 65?

---

## Task 3: Write a function that reads several columns at once

`apply(axis=1)` passes a whole row to your function, so it can read many columns
together. Labour force status is the natural case: it depends on eight answers,
and no single column expression can express it.

Because 2.6 kept the value labels, the answers are the Portuguese words `Sim` and
`Nao`, so the function reads almost like the questionnaire.

**What to do:** complete `labour_force_status` following the ILO rule:

1. under 15 years old, return `'Outside labour force'`
2. answered `Sim` to any of `worked_for_pay`, `worked_own_account` or
   `absent_from_job`, return `'Employed'`
3. answered `Sim` to `sought_job` **or** `sought_business`, **and** `Sim` to
   `available_last_week` **or** `available_next_2weeks`, return `'Unemployed'`
4. otherwise return `'Outside labour force'`

Then apply it with `axis=1` and store the result in `lf_status`.

**PT:** `apply(axis=1)` passa a linha inteira, por isso a funcao pode ler varias
colunas. Como 2.6 manteve as etiquetas, as respostas sao `Sim` e `Nao`.

**O que fazer:** complete `labour_force_status` segundo a regra da OIT: menos de
15 anos, fora da forca de trabalho; `Sim` a qualquer uma das tres perguntas de
trabalho, empregado; `Sim` a procura **e** `Sim` a disponibilidade, desempregado;
caso contrario, fora da forca de trabalho. Aplique com `axis=1` e guarde em
`lf_status`.

In [ ]:
def labour_force_status(row):
    """ILO status for one person / Situacao perante o trabalho de uma pessoa."""
    if row['age'] < 15:
        return 'Outside labour force'

    worked = (row['worked_for_pay'], row['worked_own_account'], row['absent_from_job'])
    if 'Sim' in worked:
        return 'Employed'

    searched =   # your code here: Sim to sought_job or sought_business
    available =   # your code here: Sim to either availability question
    if searched and available:
        return 'Unemployed'

    return 'Outside labour force'


df['lf_status'] = df.apply(labour_force_status, axis=1)
df['lf_status'].value_counts()

**Questions:**

- How many are employed, unemployed and outside the labour force?
- Try the availability test with `available_next_2weeks` alone. What rate do you
  get, and why is it wrong?

**PT:** Quantos empregados, desempregados e fora da forca de trabalho? Experimente
testar a disponibilidade so com `available_next_2weeks`: que taxa obtem, e porque
esta errada?

---

## Task 4: Weight the result

Each person represents many Angolans, and the `weight` column says how many. An
unweighted rate describes the sample; a weighted rate describes the country.

**What to do:** complete `weighted_share` so it returns the weighted percentage
of the population picked out by a boolean mask, then use it for the unemployment
rate, which is over the labour force, and the participation rate, which is over
the working age population.

**PT:** Cada pessoa representa muitos angolanos, e a coluna `weight` diz quantos.

**O que fazer:** complete `weighted_share` para devolver a percentagem ponderada
selecionada por uma mascara, e use-a para a taxa de desemprego, sobre a forca de
trabalho, e a taxa de atividade, sobre a populacao em idade ativa.

In [ ]:
def weighted_share(mask, weights):
    """Weighted percentage selected by `mask` / Percentagem ponderada."""
    # your code here: weighted share of the mask, as a percentage
    # o seu codigo aqui: percentagem ponderada da mascara
    return


weight = df['weight']
in_labour_force = df['lf_status'].isin(['Employed', 'Unemployed'])
working_age = df['age'] >= 15

unemployment = weighted_share(df['lf_status'] == 'Unemployed', weight[in_labour_force])
participation = weighted_share(in_labour_force, weight[working_age])

print(f'Unemployment rate:  {unemployment:5.1f}%')
print(f'Participation rate: {participation:5.1f}%')

**Questions:**

- What are the unemployment and participation rates?
- Is this the strict or the relaxed definition?

**PT:** Quais sao as taxas de desemprego e de atividade? Esta e a definicao
estrita ou a alargada?

---

# Part B: the trade workbooks

## Task 5: Load two trade sheets and tidy their column names

INE publishes the trade data as spreadsheets made for human readers: two title
rows above the header, a blank row, a `Total Geral` row, the data, and a source
footer at the bottom.

**What to do:**

1. load the export sheet and the import sheet with `skiprows=2`, so the real
   header becomes the header, into `export_df` and `import_df`
2. print the column names of both and look at what arrived
3. convert those names to snake case, then keep only the rows where the country
   name is filled in, which drops the blank row, the total and the footer at once

**PT:** O INE publica os dados em folhas feitas para leitura humana: duas linhas
de titulo, o cabecalho, uma linha vazia, o `Total Geral`, os dados, e o rodape.

**O que fazer:** carregue as duas folhas com `skiprows=2` para `export_df` e
`import_df`; imprima os nomes das colunas; converta esses nomes para snake case e
mantenha so as linhas com o nome do pais preenchido.

In [ ]:
PARTNERS_FILE = 'Comercio Externo de Bens por Países Parceiros.xlsx'
partners_path = os.path.join(trade_dir, PARTNERS_FILE)

export_df = pd.read_excel(partners_path, sheet_name='Exportação por Países (USD)',
                          skiprows=2)
import_df = pd.read_excel(partners_path, sheet_name='Importação por Países (USD)',
                          skiprows=2)

print('export_df:', export_df.shape, '| import_df:', import_df.shape)

In [ ]:
# Look at the names before touching them / Ver os nomes antes de mexer
print(list(export_df.columns))

In [ ]:
def to_snake_case(columns):
    """Lower case, strip accents, and join words with underscores.

    Minusculas, sem acentos, e palavras unidas por underscore.
    """
    return (columns
            .str.replace('\n', ' ', regex=False)
            .str.strip()
            .str.lower()
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.replace(' ', '_', regex=False))


export_df.columns = to_snake_case(export_df.columns)
import_df.columns = to_snake_case(import_df.columns)

# Rows without a country name are the blank row, the total and the footer
# As linhas sem nome de pais sao a linha vazia, o total e o rodape
export_df =   # your code here: keep the rows where pais is filled in
import_df =   # your code here: o mesmo para as importacoes

print(list(export_df.columns)[:4], '...', list(export_df.columns)[-1:])
print('export_df:', export_df.shape, '| import_df:', import_df.shape)
export_df[['codigo', 'pais', 'ano_2025']].head()

**Questions:**

- What did the raw column names look like, and what was hiding inside the year
  names?
- What are they after tidying?
- How many rows does each sheet give? Which one is not a country?
- What unit are the values in, and where does the file say so?

**PT:** Como eram os nomes originais e o que estava escondido nos nomes dos anos?
E depois de arrumados? Quantas linhas tem cada folha e qual nao e um pais? Em que
unidade estao os valores?

---

## Task 6: Merge the two flows and classify each partner

Both tables have one row per country, so this is a one to one merge.

**What to do:**

1. merge `export_df` and `import_df` on `codigo`, keeping both sides, with
   `indicator=True` and `validate='one_to_one'`, renaming the two value columns
   to `exports_thousand_usd` and `imports_thousand_usd`, and `codigo` and `pais`
   to `country_code` and `country_name`
2. compute `balance_thousand_usd` as exports minus imports, treating a missing
   flow as zero
3. write `partner_profile`, which needs both value columns at once and therefore
   runs with `axis=1`

**PT:** As duas tabelas tem uma linha por pais, por isso a juncao e um para um.

**O que fazer:** junte `export_df` e `import_df` por `codigo` com
`indicator=True` e `validate='one_to_one'`; calcule `balance_thousand_usd`; e
escreva `partner_profile`, que precisa das duas colunas ao mesmo tempo e corre
com `axis=1`.

In [ ]:
YEAR = 'ano_2025'

# your code here / o seu codigo aqui
trade = pd.merge(

print(trade['_merge'].value_counts())
trade = trade.drop(columns='_merge')
print('Merged:', trade.shape)

In [ ]:
def partner_profile(row):
    """Describe Angola's 2025 relationship with one partner.

    Return, in this order of priority:
      'Incomplete'          when either flow is missing
      'Negligible'          when the two flows together are under 1000
      'Angola mainly sells' when exports are more than double imports
      'Angola mainly buys'  when imports are more than double exports
      'Two way'             otherwise

    Devolver, por esta ordem de prioridade: 'Incomplete' se faltar um fluxo,
    'Negligible' se a soma for inferior a 1000, 'Angola mainly sells' se as
    exportacoes forem mais do dobro das importacoes, 'Angola mainly buys' no caso
    inverso, e 'Two way' nos restantes casos.
    """
    # your code here / o seu codigo aqui
    return


trade['balance_thousand_usd'] =   # your code here: exports minus imports,
                               #                 a missing flow counting as 0
trade['profile'] =   # your code here: apply partner_profile with axis=1

print(trade['profile'].value_counts())

In [ ]:
print('Largest surpluses / Maiores excedentes:')
print(trade.nlargest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))
print()
print('Largest deficits / Maiores defices:')
print(trade.nsmallest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))

**Questions:**

- Did every partner match, and what does `validate='one_to_one'` promise?
- Which partners show the largest surplus and deficit?
- What would happen to the profiles without the `Negligible` threshold?

**PT:** Todos os parceiros corresponderam? Que parceiros tem maior excedente e
defice? O que aconteceria sem o limiar `Negligible`?

---

## Task 7: Do not sum a hierarchy by accident

This second workbook breaks exports down by economic category. Its code column
holds a tree: a one digit code is a section, two digits a group inside that
section, three digits a subgroup inside that group.

So the rows are not comparable. Adding them all up counts the same money more
than once. The sheet publishes its own `Total Geral`, which is the check.

**What to do:** load and tidy this sheet the same way as before, look at the
codes, set the published total aside, then write `cgce_level` to measure how deep
each code sits and use it to find which rows may be summed.

**PT:** Este segundo ficheiro reparte as exportacoes por categoria economica. A
coluna de codigo guarda uma arvore: um digito e uma seccao, dois digitos um grupo
dentro dela, tres digitos um subgrupo. As linhas nao sao comparaveis, e soma-las
todas conta o mesmo dinheiro mais do que uma vez.

**O que fazer:** carregue e arrume esta folha como antes, veja os codigos, guarde
o total publicado, e escreva `cgce_level` para medir a profundidade de cada
codigo.

In [ ]:
CGCE_FILE = 'Comercio Externo de Bens por Grandes Categorias Económicas.xlsx'
cgce_path = os.path.join(trade_dir, CGCE_FILE)

cgce_df = pd.read_excel(cgce_path, sheet_name='Export Cat. Económica (USD)',
                        skiprows=2)
cgce_df.columns =   # your code here: reuse to_snake_case
cgce_df =   # your code here: keep the rows where descricao is filled in

print('cgce_df:', cgce_df.shape)
print(list(cgce_df.columns)[:4])

In [ ]:
# The codes get longer as the categories get narrower
# Os codigos ficam mais longos a medida que as categorias se estreitam
cgce_df[['cgce', 'descricao', YEAR]].head(8)

In [ ]:
# Keep the published total aside, then drop that row from the data
# Guardar o total publicado e remover essa linha dos dados
published_total = cgce_df.loc[cgce_df['descricao'] == 'Total Geral', YEAR].iloc[0]
cgce_df = cgce_df[cgce_df['cgce'].notna()]

print('Published Total Geral:', f'{published_total:,.0f}')
print('Category rows:', len(cgce_df))

In [ ]:
def cgce_level(code):
    """How deep a code sits: 1 section, 2 group, 3 subgroup.

    Profundidade do codigo: 1 seccao, 2 grupo, 3 subgrupo.
    """
    # your code here: the length of the code / o comprimento do codigo
    return


cgce_df['level'] = cgce_df['cgce'].apply(cgce_level)
print(cgce_df['level'].value_counts().sort_index())

In [ ]:
naive =   # your code here: sum every row
sections_only =   # your code here: sum only the level 1 rows

print(f'Published total:     {published_total:15,.0f}')
print(f'Sum of every row:    {naive:15,.0f}  <- {naive / published_total:.2f}x')
print(f'Sum of level 1 only: {sections_only:15,.0f}')
print()
print('Level 1 matches the published total:',
      bool(abs(sections_only - published_total) < 1))

**Questions:**

- How many rows sit at each level of the tree?
- Compare the sum of every row with the published `Total Geral`. What is the
  ratio, and why is it exactly that?
- Which rows do you sum to reproduce the published figure?

**PT:** Quantas linhas em cada nivel? Compare a soma de todas as linhas com o
`Total Geral` publicado: qual e a razao e porque e exatamente essa? Que linhas
deve somar?

---

## Task 8: Save

**What to do:** write the survey and the trade table to `20_processed/` with
`index=False`.

**PT:** **O que fazer:** grave o inquerito e a tabela de comercio em
`20_processed/` com `index=False`.

In [ ]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)

survey_out = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')
trade_out = os.path.join(DATA_PROC_DIR, 'angola_trade_partners.csv')

# your code here: write both frames with index=False
# o seu codigo aqui: gravar as duas tabelas com index=False

print('survey:', df.shape, '| trade:', trade.shape)

**Questions:**

- Which columns did the survey gain, and which function produced each?

**PT:** Que colunas ganhou o inquerito, e que funcao produziu cada uma?